<a href="https://colab.research.google.com/github/papadizzle/Fantasy_Head_Coach/blob/main/Fantasy_Head_Coach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

crewai: Framework for orchestrating multi-agent systems.

langchain-google-genai: Connects CrewAI directly to Gemini.

yfpy: Popular Python wrapper for the Yahoo Fantasy Sports API.

In [ ]:
!pip install crewai langchain-google-genai yfpy


Import Gemini & Yahoo! secret keys from CoLab

In [ ]:
import os
from google.colab import userdata

# Fetch keys from Colab Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
YAHOO_CONSUMER_KEY = userdata.get('YAHOO_CONSUMER_KEY')
YAHOO_CONSUMER_SECRET = userdata.get('YAHOO_CONSUMER_SECRET')

# Set as environment variables for use by CrewAI/LiteLLM and yfpy
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
os.environ['YAHOO_CONSUMER_KEY'] = YAHOO_CONSUMER_KEY
os.environ['YAHOO_CONSUMER_SECRET'] = YAHOO_CONSUMER_SECRET

print("Keys successfully loaded into the session environment.")


Connect to Yahoo! Fantasy API

In [ ]:
import os
import json
import urllib.parse
from yfpy.query import YahooFantasySportsQuery

auth_dir = "."
private_file = os.path.join(auth_dir, "private.json")

consumer_key = os.environ.get('YAHOO_CONSUMER_KEY')
consumer_secret = os.environ.get('YAHOO_CONSUMER_SECRET')

# Step 1: Write private.json if it doesn't exist
if not os.path.exists(private_file):
    creds = {
        "consumer_key": consumer_key,
        "consumer_secret": consumer_secret
    }
    with open(private_file, "w") as f:
        json.dump(creds, f)

# Step 2: Print authorization link directly
params = {
    "client_id": consumer_key,
    "response_type": "code",
    "redirect_uri": "oob"
}
auth_url = f"https://api.login.yahoo.com/oauth2/request_auth?{urllib.parse.urlencode(params)}"

print("\n------------------------------------------------------------")
print("1. CLICK THIS LINK TO AUTHORIZE YAHOO:")
print(auth_url)
print("------------------------------------------------------------\n")

# Step 3: Let yfpy handle authorization natively in a single prompt
LEAGUE_ID = "1335847"

yahoo_query = YahooFantasySportsQuery(
    auth_dir=auth_dir,
    league_id=LEAGUE_ID,
    game_code="nfl"
)

print("\nSuccessfully authenticated and initialized Yahoo Query!")

# Step 4: Resolve YOUR team_id and the CURRENT week dynamically instead of
# hardcoding them. Fall back to safe defaults if the lookup fails, since
# exact method names can vary slightly by yfpy version -- check yfpy's docs
# for your installed version if these calls error out.
MY_TEAM_ID = "1"
CURRENT_WEEK = 1

try:
    user_teams = yahoo_query.get_user_teams()
    if user_teams:
        MY_TEAM_ID = str(user_teams[0].team_id)
        print(f"Resolved your team_id: {MY_TEAM_ID}")
except Exception as e:
    print(f"Could not auto-resolve team_id, defaulting to '{MY_TEAM_ID}'. Error: {e}")

try:
    league_info = yahoo_query.get_league_info()
    CURRENT_WEEK = int(getattr(league_info, "current_week", CURRENT_WEEK))
    print(f"Resolved current week: {CURRENT_WEEK}")
except Exception as e:
    print(f"Could not auto-resolve current week, defaulting to week {CURRENT_WEEK}. Error: {e}")


In [ ]:
import os

print(f"YAHOO_CONSUMER_KEY (masked): {os.environ.get('YAHOO_CONSUMER_KEY', '')[:8]}...{os.environ.get('YAHOO_CONSUMER_KEY', '')[-8:]}")
print(f"YAHOO_CONSUMER_SECRET (masked): {os.environ.get('YAHOO_CONSUMER_SECRET', '')[:8]}...{os.environ.get('YAHOO_CONSUMER_SECRET', '')[-8:]}")

### Re-authenticating with Yahoo Fantasy Sports API

The `401 Client Error: Unauthorized` indicates an issue with the Yahoo API token. To resolve this, we need to delete the `token.json` file (which stores your Yahoo API credentials) and then re-run the previous cell (`BZplraFN3yVS`) to initiate a fresh authentication process.

In [ ]:
import os

# Delete token.json to force a fresh authentication flow
if os.path.exists('token.json'):
    os.remove('token.json')
    print("Deleted 'token.json'. Please re-run the previous cell to re-authenticate with Yahoo.")
else:
    print("'token.json' not found. If you are still encountering authentication issues, please ensure your consumer key and secret are correct and re-run the authentication cell.")

Environment Setup & LLM Configuration

In [ ]:
import logging
from crewai import Agent, Crew, Process, Task, LLM

# Quiet down noisy HTTP/debug logs so crew output is readable
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# GEMINI_API_KEY is already in os.environ from the secrets cell above --
# no need to set it again here, and definitely never hardcode it in the notebook.

# Instantiate the LLM using CrewAI's explicit native wrapper
gemini_llm = LLM(
    model="gemini/gemini-3.6-flash",  # Format MUST be provider/model-name
    temperature=0.2,
)


TEST KEY

In [ ]:
print(f"Key length: {len(os.environ.get('GEMINI_API_KEY', ''))} | Starts with: {os.environ.get('GEMINI_API_KEY', '')[:6]}")


Custom Tools Definition

In [ ]:
from crewai.tools import tool

@tool("Fetch User Roster")
def get_roster_tool(team_id: str) -> str:
    """Retrieves current roster players, positions, and injury statuses for a team."""
    try:
        roster = yahoo_query.get_team_roster_by_week(team_id=team_id, week=CURRENT_WEEK)
        return str([
            f"{p.name.full} ({p.display_position}) - "
            f"Status: {getattr(p, 'status', 'Healthy') or 'Healthy'}, "
            f"Selected Position: {getattr(getattr(p, 'selected_position', None), 'position', 'N/A')}"
            for p in roster
        ])
    except Exception as e:
        return f"Error fetching roster: {e}"

@tool("Fetch Available Free Agents")
def get_free_agents_tool(position: str) -> str:
    """Retrieves top available free agents in the league by position, with team, ownership, and injury status."""
    try:
        fa = yahoo_query.get_league_free_agents(position=position)
        return str([
            f"{p.name.full} ({p.display_position}) - "
            f"Team: {getattr(p, 'editorial_team_abbr', 'N/A')}, "
            f"Pct Owned: {getattr(p, 'percent_owned', 'N/A')}, "
            f"Status: {getattr(p, 'status', 'Healthy') or 'Healthy'}"
            for p in fa[:10]
        ])
    except Exception as e:
        return f"Error fetching free agents: {e}"


Agent Instantiation

In [ ]:
from crewai import Agent

# 1. Waiver Wire Specialist
waiver_scout = Agent(
    role="Waiver Wire & Talent Scout",
    goal="Identify high-upside free agents based on usage, injuries, and available talent",
    backstory="You are an expert fantasy analyst who specializes in finding league-winning waiver pickups before anyone else.",
    tools=[get_free_agents_tool],
    verbose=True,
    llm=gemini_llm
)

# 2. Roster Optimization Specialist
lineup_optimizer = Agent(
    role="Head Coach & Lineup Optimizer",
    goal="Analyze roster strengths, weaknesses, and bench players to select starting lineups, factoring in any waiver pickups the scout recommends",
    backstory="You are a data-driven fantasy head coach who optimizes start/sit decisions to maximize weekly ceiling and floor.",
    tools=[get_roster_tool],
    verbose=True,
    llm=gemini_llm
)


Task Definitions

In [ ]:
from crewai import Task

scout_task = Task(
    description=(
        "Use the 'Fetch Available Free Agents' tool to scan available 'RB' and 'WR' positions. "
        "Identify top 3 additions for PPR scoring based on available players, their ownership "
        "percentage, and injury status."
    ),
    expected_output="A list of top 3 waiver pickups with brief reasoning for each.",
    agent=waiver_scout,
    async_execution=True,  # doesn't depend on the roster lookup, so it can run in parallel
)

lineup_task = Task(
    description=(
        f"Use the 'Fetch User Roster' tool to examine team roster for team_id '{MY_TEAM_ID}'. "
        "Compare current starters against bench targets and suggest start/sit adjustments. "
        "Take the waiver wire scout's recommendations into account -- if a suggested pickup "
        "would clearly outperform a current bench or borderline starter, factor that into your call."
    ),
    expected_output="Recommended starting lineup with concise start/sit explanations.",
    agent=lineup_optimizer,
    context=[scout_task],
)

final_report_task = Task(
    description=(
        "Combine the waiver wire scout's pickups and the lineup optimizer's start/sit "
        "recommendations into a single, cohesive weekly game plan a fantasy manager can act on "
        "in under a minute of reading."
    ),
    expected_output=(
        "A short weekly report with two sections: 'Waiver Wire Moves' and 'Starting Lineup', "
        "each with 1-2 sentences of reasoning per recommendation."
    ),
    agent=lineup_optimizer,
    context=[scout_task, lineup_task],
)


Crew Setup & Async Execution

In [ ]:
from crewai import Crew, Process
from datetime import datetime

# Assemble the Crew
fantasy_crew = Crew(
    agents=[waiver_scout, lineup_optimizer],
    tasks=[scout_task, lineup_task, final_report_task],
    process=Process.sequential,
    memory=True,  # lets agents recall context across weekly runs
    verbose=True,
)

# Define an async function to execute the crew
async def run_crew():
    result = await fantasy_crew.kickoff_async()
    print("\n=== AGENT RECOMMENDATIONS ===")
    print(result.raw)

    # Persist this week's report so you can track recommendations over time
    filename = f"fantasy_report_week{CURRENT_WEEK}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    with open(filename, "w") as f:
        f.write(f"# Fantasy Head Coach Report - Week {CURRENT_WEEK}\n\n")
        f.write(result.raw)
    print(f"\nSaved report to {filename}")

# Kick off the async function
await run_crew()
